# Label Studio workflow for ERPgnostics-style classes

This notebook prepares Label Studio tasks from the existing manifest and exports
final labels as a simple CSV (channel, sort_col, class) that Julia can read.


In [1]:
from pathlib import Path
import csv
import json
from urllib.parse import urlparse, unquote

# Paths
REPO_ROOT = Path.cwd()
NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "model_test"
if not NOTEBOOK_DIR.exists():
    NOTEBOOK_DIR = Path.cwd()

EXPORT_DIR = NOTEBOOK_DIR / "erp_labelstudio_exports"
MANIFEST_PATH = EXPORT_DIR / "labelstudio_manifest.csv"

LABEL_CONFIG_PATH = NOTEBOOK_DIR / "labelstudio_config.xml"
TASKS_JSON_PATH = EXPORT_DIR / "labelstudio_tasks.json"

# Expected Label Studio export (set this after exporting)
EXPORT_JSON_PATH = EXPORT_DIR / "labelstudio_export.json"

OUT_CSV_PATH = EXPORT_DIR / "labelstudio_labels_for_julia.csv"
OUT_JSON_PATH = EXPORT_DIR / "labelstudio_labels_for_julia.json"

print("Notebook dir:", NOTEBOOK_DIR)
print("Export dir:", EXPORT_DIR)
print("Manifest:", MANIFEST_PATH)


Notebook dir: /home/benjamin/Dokumente/BA2/notebooks/model_test
Export dir: /home/benjamin/Dokumente/BA2/notebooks/model_test/erp_labelstudio_exports
Manifest: /home/benjamin/Dokumente/BA2/notebooks/model_test/erp_labelstudio_exports/labelstudio_manifest.csv


In [2]:
# ERPgnostics classes (6 patterns + 1 extra)
# Change CO_CLASS_NAME to "no_class" if you want to align with existing Julia training code.
CO_CLASS_NAME = "co_class"

CLASS_NAMES = [
    "sigmoid",
    "one_sided_fan",
    "two_sided_fan",
    "diverging_bar",
    "hourglass",
    "tilted_bar",
    CO_CLASS_NAME,
]

CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

# Aliases in case a different spelling appears in Label Studio exports
LABEL_ALIASES = {
    "no_class": CO_CLASS_NAME,
    "noclass": CO_CLASS_NAME,
    "co-class": CO_CLASS_NAME,
}

print("Classes:", CLASS_NAMES)


Classes: ['sigmoid', 'one_sided_fan', 'two_sided_fan', 'diverging_bar', 'hourglass', 'tilted_bar', 'co_class']


In [3]:
def load_manifest(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Manifest not found: {path}")

    rows = []
    with path.open(newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_path = Path(row["image"]).expanduser().resolve()
            rows.append({
                "image": str(image_path),
                "channel": int(row["channel"]),
                "sort_col": row["sort_col"].strip(),
            })
    return rows

manifest_rows = load_manifest(MANIFEST_PATH)
print("Loaded", len(manifest_rows), "rows from manifest")
print("Example:", manifest_rows[0])


Loaded 100 rows from manifest
Example: {'image': '/home/benjamin/Dokumente/BA2/notebooks/model_test/erp_labelstudio_exports/ch21__sort-duration.png', 'channel': 21, 'sort_col': 'duration'}


In [4]:
def path_to_file_uri(path_str: str) -> str:
    return Path(path_str).resolve().as_uri()

# Label Studio labeling config
label_config = '''
<View>
  <Image name="image" value="$image"/>
  <Choices name="pattern" toName="image" choice="single" showInLine="true">
    {choices}
  </Choices>
  <Text name="meta" value="channel: $channel | sort: $sort_col"/>
</View>
'''.strip()

choices_xml = "\n".join([f'    <Choice value="{c}"/>' for c in CLASS_NAMES])
label_config = label_config.format(choices=choices_xml)

LABEL_CONFIG_PATH.write_text(label_config)
print("Wrote Label Studio config to:", LABEL_CONFIG_PATH)

# Build tasks JSON for Label Studio import
# We include both file URI and raw path for easier round-tripping.

tasks = []
for i, row in enumerate(manifest_rows, start=1):
    tasks.append({
        "id": i,
        "data": {
            "image": path_to_file_uri(row["image"]),
            "image_path": row["image"],
            "channel": row["channel"],
            "sort_col": row["sort_col"],
        },
    })

with TASKS_JSON_PATH.open("w") as f:
    json.dump(tasks, f, indent=2)

print("Wrote", len(tasks), "tasks to:", TASKS_JSON_PATH)


Wrote Label Studio config to: /home/benjamin/Dokumente/BA2/notebooks/model_test/labelstudio_config.xml
Wrote 100 tasks to: /home/benjamin/Dokumente/BA2/notebooks/model_test/erp_labelstudio_exports/labelstudio_tasks.json


## Label Studio steps

1. Create a new project in Label Studio.
2. Paste the content of `labelstudio_config.xml` into the labeling interface.
3. Import `labelstudio_tasks.json`.
4. Label all tasks and export **JSON** to `labelstudio_export.json` in the same folder.


In [5]:
def file_uri_to_path(uri: str) -> str:
    if uri.startswith("file:"):
        parsed = urlparse(uri)
        return unquote(parsed.path)
    return uri


def extract_label_from_result(result: dict):
    rtype = result.get("type")
    value = result.get("value", {})
    if rtype == "choices":
        choices = value.get("choices") or []
        return choices[0] if choices else None
    if rtype == "labels":
        labels = value.get("labels") or []
        return labels[0] if labels else None
    return None


def pick_annotation(annotations):
    if not annotations:
        return None
    # Prefer non-draft, non-cancelled annotations
    clean = [a for a in annotations if not a.get("was_cancelled") and not a.get("draft")]
    if clean:
        annotations = clean
    # Use latest by created_at if available
    def key(a):
        return a.get("created_at", "")
    annotations = sorted(annotations, key=key)
    return annotations[-1]


def load_labelstudio_export(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Export not found: {path}")
    with path.open() as f:
        data = json.load(f)
    if isinstance(data, dict) and "tasks" in data:
        return data["tasks"]
    if isinstance(data, list):
        return data
    raise ValueError("Unexpected Label Studio export format")


def normalize_label(label: str):
    if label is None:
        return None
    label = label.strip()
    if label in CLASS_TO_ID:
        return label
    return LABEL_ALIASES.get(label, label)


def build_labels_from_export(tasks):
    rows = []
    unknown_labels = set()
    missing_label = 0

    for task in tasks:
        data = task.get("data", {})
        image_path = data.get("image_path")
        if not image_path:
            image_uri = data.get("image") or ""
            image_path = file_uri_to_path(image_uri)

        channel = data.get("channel")
        sort_col = data.get("sort_col")

        annotations = task.get("annotations") or task.get("completions") or []
        annotation = pick_annotation(annotations)
        label = None
        if annotation:
            for res in annotation.get("result", []):
                label = extract_label_from_result(res)
                if label:
                    break

        label = normalize_label(label)

        if label is None:
            missing_label += 1
        elif label not in CLASS_TO_ID:
            unknown_labels.add(label)

        rows.append({
            "image": image_path,
            "channel": channel,
            "sort_col": sort_col,
            "class_name": label,
            "class_id": CLASS_TO_ID.get(label),
        })

    return rows, missing_label, unknown_labels


if EXPORT_JSON_PATH.exists():
    tasks = load_labelstudio_export(EXPORT_JSON_PATH)
    label_rows, missing_label, unknown_labels = build_labels_from_export(tasks)

    # Write CSV for Julia
    with OUT_CSV_PATH.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "channel", "sort_col", "class_name", "class_id"])
        writer.writeheader()
        writer.writerows(label_rows)

    # Write JSON as an alternative
    with OUT_JSON_PATH.open("w") as f:
        json.dump(label_rows, f, indent=2)

    print("Wrote:", OUT_CSV_PATH)
    print("Wrote:", OUT_JSON_PATH)
    print("Missing labels:", missing_label)
    if unknown_labels:
        print("Unknown labels:", sorted(unknown_labels))
else:
    print("Export JSON not found yet:", EXPORT_JSON_PATH)
    print("Run Label Studio export and save JSON to that path.")


Export JSON not found yet: /home/benjamin/Dokumente/BA2/notebooks/model_test/erp_labelstudio_exports/labelstudio_export.json
Run Label Studio export and save JSON to that path.
